<a href="https://colab.research.google.com/github/cmiyachi/AI-Yogini-Project/blob/master/Another_copy_of_multimodal_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 5: **Build a Multi-Modal Generation Agent**

Welcome to the final project! In this project, you'll use open-source text-to-image and text-to-video models to generate content. Next, you'll build a **unified multi-modal agent** similar to modern chatbots, where a single agent can support general questions, image generation, and video generation requests.

By the end of this project, you'll understand how to integrate multiple model types under one  routing system capable of deciding what modality to use based on the user's intent.



## Learning Objectives

* Use **Text-to-Image** models to generate images from a text.
* Generate short clips with a **Text-to-Video** model
* Build a **Multi-Modal Agent** that answers questions and routes media requests
* Build a simple **Gradio** UI and interact with the multi-modal agent

## Roadmap
1. Environment setup
2. Text‑to‑Image
3. Text‑to‑Video
4. Multimodal Agent
5. Gradio UI
6. Celebrate

## 1 - Environment Setup

In this project, we'll use open-source Text-to-Image and Text-to-Video models to generate visuals from natural-language prompts. These models are computationally heavy and perform best on GPUs, so we recommend running this notebook in Google Colab or another GPU-enabled environment. We'll load all models from Hugging Face, which requires authentication.

Before continuing:
1. Open this project in Google Colab. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bytebyteai/ai-eng-projects/blob/main/project_5/multimodal_agent.ipynb)
2. Create a Hugging Face account and generate an access token at huggingface.co/settings/tokens
3. Paste your token in the field below to log in.
4. In the Colab environment, enable GPU acceleration by selecting Runtime → Change runtime type → GPU.

In [ ]:
from huggingface_hub import login

login(token="")

Let's import the required libraries and confirm that PyTorch can detect the available GPU.

In [ ]:
import torch, diffusers, transformers, os, random, gc
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2 - Text-to-Image (T2I)
T2I models translate natural-language descriptions into images. They are typically based on diffusion models, which gradually refine random noise into a coherent picture guided by the text prompt. In this section, you'll load and test one such model to generate images directly from text inputs.

### 2.1: Load a T2I Model
We'll use `Stable Diffusion XL` (SDXL) by `Stability AI`, one of the open-source diffusion models. It provides high-quality, detailed image generation with relatively efficient inference compared to earlier versions.

You'll load the model from Hugging Face using the diffusers library, which simplifies running diffusion-based pipelines. To learn more about diffusers, read: https://huggingface.co/docs/diffusers/main/index


In [ ]:
from diffusers import DiffusionPipeline

# Define the Stable Diffusion XL model ID from Hugging Face and load the pre-trained model
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipeline = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)
pipeline.to("cuda")

### 2.2: Generate an image

In [ ]:
# Generate and display an image from a text prompt using the loaded pipeline
prompt = "a clear realistic mountain scene of a female snowboardsnowboarding in powder"
image = pipeline(prompt).images[0]
image.save("astronaut_on_mars.png")
display(image)

### 2.3: Experimenting with "inference_steps"

The number of inference steps determines how many refinement passes the diffusion model makes. Fewer steps give quicker but less detailed images, while more steps improve clarity and structure at the cost of speed.

Try generating images with different step counts and compare the results.

In [ ]:
import matplotlib.pyplot as plt

# Generate an image for different values of num_inference_steps (e.g., 10, 25, 50) and compare sharpness and detail
images = []

# Define the prompt for image generation
custom_prompt = "a clear realistic mountain scene of a female snowboarder snowboarding in powder"

# Experiment with different numbers of inference steps
for steps in [10, 25, 50]:
    image = pipeline(custom_prompt, num_inference_steps=steps).images[0]
    images.append((steps, image))

# Plot results side-by-side
plt.figure(figsize=(12, 4))
for i, (steps, img) in enumerate(images, 1):
    plt.subplot(1, len(images), i)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{steps} steps")
plt.tight_layout()
plt.show()

### 2.4 (Optional): Visualizing the Diffusion Process
Diffusion models start from random noise and iteratively refine it into an image that matches the prompt. If you are curious, visualize all intermediate steps to see how the noise gradually turns into a coherent picture.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

intermediate_images = []
num_inference_steps = 50
custom_prompt = "a clear realistic mountain scene of a female snowboarder snowboarding in powder"

def store_intermediate_image(pipe, step_idx, timestep, callback_kwargs):
    # Get latents from callback_kwargs
    latents = callback_kwargs["latents"]

    # Decode latents to image space (tensor in range [-1, 1])
    image_tensor = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[0]

    # Use the pipeline's image processor to postprocess and convert to PIL Image
    # This method is designed to handle denormalization and type conversion robustly.
    image = pipe.image_processor.postprocess(image_tensor, output_type="pil")[0]
    intermediate_images.append(image)
    return callback_kwargs

# Run the pipeline with callback
pipeline(
    custom_prompt,
    num_inference_steps=num_inference_steps,
    callback_on_step_end=store_intermediate_image,
    output_type="latent" # Crucial for `latents` to be in callback_kwargs
)

# Plot a selection of the captured images
plt.figure(figsize=(15, 3))
num_to_display = 5
interval = max(1, len(intermediate_images) // num_to_display)

for i in range(num_to_display):
    idx = i * interval
    if idx < len(intermediate_images):
        plt.subplot(1, num_to_display, i + 1)
        plt.imshow(intermediate_images[idx])
        plt.title(f"Step {idx}")
        plt.axis("off")
plt.tight_layout()
plt.show()

### 2.5 (Optional): Experiment with other models.
Different text-to-image models vary in speed, style, and visual quality. Try swapping in other open-source diffusion models and compare how their outputs differ in detail, realism, or artistic tone.

You can browse available models on Hugging Face here: https://huggingface.co/models?library=diffusers

In [ ]:
# Step 1: Replace model_id with another text-to-image model from Hugging Face
# Step 2: Reload the pipeline and generate a few test images
# Step 3: Compare image quality, color balance, and prompt fidelity

# Example: Swap to Stable Diffusion v1.5
new_model_id = "runwayml/stable-diffusion-v1-5"

# Clear previous pipeline to free up GPU memory if needed (optional but good practice)
if 'pipeline' in locals() and pipeline is not None:
    print("Moving previous pipeline to CPU and clearing cache...")
    pipeline.to("cpu") # Explicitly move to CPU to free VRAM
    del pipeline
    torch.cuda.empty_cache()
    gc.collect() # Force garbage collection

pipeline = DiffusionPipeline.from_pretrained(
    new_model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)
pipeline.to("cuda")

# Generate a test image with the new model
new_prompt = "a clear realistic mountain scene of a female snowboarding in powder"
new_image = pipeline(new_prompt).images[0]
print(f"Generated image using {new_model_id} with prompt: '{new_prompt}'")
display(new_image)

## 3 - Text-to-Video (T2V)
T2V models extend the idea of diffusion from still images to moving sequences. Instead of generating one frame, they create a series of coherent frames that depict motion consistent with the text prompt. These models are computationally heavier and often generate short clips (typically 2-10 seconds).

In this section, you'll load an open-source video diffusion model and prepare it for generation.

### 3.1: Load a T2V model

We'll use the model `damo-vilab/text-to-video-ms-1.7b`, which can produce short video clips from text prompts. This model benefits from a specialized scheduler (DPMSolverMultistepScheduler) that improves stability and speed during sampling.

In [ ]:
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

video_model_id = 'damo-vilab/text-to-video-ms-1.7b'

# Load the model with FP16 precision for efficiency
pipeline = DiffusionPipeline.from_pretrained(video_model_id, torch_dtype=torch.float16, variant="fp16")
pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
pipeline.to("cuda")

### 3.2: Generate a clip
Create a short video clip from a text prompt using a text-to-video model.

In [ ]:
# Step 1: Write a text prompt describing the video you want to generate
video_prompt = "a dog walking in a park, 4k, high resolution"
# Step 2: Run the text-to-video pipeline with your chosen prompt
vid_frames = pipeline(video_prompt).frames

### 3.3: Frame inspection
Inspect a single frame to sanity-check colors, resolution, and subject positioning before writing a full video.

In [ ]:
import numpy as np
from PIL import Image

# Step 1: Select one frame from vid_frames (e.g., index 0)
# Assuming vid_frames is (batch_size, num_frames, height, width, channels)
# We need to select the first frame from the first video in the batch
frame_to_inspect = vid_frames[0][0]

# Step 2: Convert float [0,1] frame to uint8 [0,255]
frame_uint8 = (frame_to_inspect * 255).astype(np.uint8)

# Step 3: Display as a PIL image
image = Image.fromarray(frame_uint8)
display(image)

### 3.4: Convert frames to MP4
Write the generated frames to an MP4 file so you can preview and share the result.

In [ ]:
# Step 1: Use diffusers.utils.export_to_video to write vid_frames to an MP4
# Step 2: Capture and print the saved video path
from diffusers.utils import export_to_video

# Assuming vid_frames is (batch_size, num_frames, height, width, channels)
# We need to pass the actual video frames (num_frames, height, width, channels)
video_path = export_to_video(vid_frames[0])
print(f"Video saved to: {video_path}")

### 3.5: Video inspection
Play the saved video inside the notebook to check motion and temporal consistency.

In [ ]:
# Display the saved MP4 inline
from IPython.display import Video

Video(video_path)

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

print(f"Shape of vid_frames: {vid_frames.shape}")
print(f"Min pixel value: {np.min(vid_frames)}")
print(f"Max pixel value: {np.max(vid_frames)}")

if vid_frames.shape[1] > 0:
    # Display the first frame
    first_frame = vid_frames[0][0]
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(first_frame)
    plt.title("First Frame")
    plt.axis('off')

    # Display the last frame
    last_frame = vid_frames[0][-1]
    plt.subplot(1, 2, 2)
    plt.imshow(last_frame)
    plt.title("Last Frame")
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("No frames were generated. vid_frames is empty.")

### 3.6 (Optional): Experiment with different configs
Increase `num_frames` or decrease `num_inference_steps` to experiment with clip length versus quality.

## 4 - Multimodal Generation Agent
Now that you have text-to-image, text-to-video, and basic LLM question answering, you will build a single agent that routes user requests to the right capability. The agent will read a prompt, infer intent (chat vs image vs video), and return the appropriate output.

### 4.1: Load an LLM for generic queries
Use a small LLM as the default chat brain. We will start with `gemma-3-1b-it` and keep the loading logic simple. You can swap to another compact chat model later.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch, textwrap, json, re

# Load google/gemma-3-1b-it using Hugging Face (NOTE: Requires explicit access from Hugging Face)
# For now, we will use an alternative open model: TinyLlama/TinyLlama-1.1B-Chat-v1.0

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)
model.to("cuda")

### 4.2: Build a routing mechanism to route requests

In [ ]:
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
import gc # Import garbage collector for memory management

# Global variables for caching pipelines to avoid re-loading frequently
_t2i_pipeline = None
_t2v_pipeline = None

def generate_media(prompt: str, mode: str):
    global _t2i_pipeline, _t2v_pipeline

    # Ensure other pipeline is unloaded to free GPU memory
    if mode == "image" and _t2v_pipeline is not None:
        print("Unloading Text-to-Video pipeline...")
        _t2v_pipeline.to("cpu")
        del _t2v_pipeline
        _t2v_pipeline = None
        torch.cuda.empty_cache()
        gc.collect()
    elif mode == "video" and _t2i_pipeline is not None:
        print("Unloading Text-to-Image pipeline...")
        _t2i_pipeline.to("cpu")
        del _t2i_pipeline
        _t2i_pipeline = None
        torch.cuda.empty_cache()
        gc.collect()

    if mode == "image":
        if _t2i_pipeline is None:
            print("Loading Text-to-Image pipeline...")
            t2i_model_id = "stabilityai/stable-diffusion-xl-base-1.0"
            _t2i_pipeline = DiffusionPipeline.from_pretrained(
                t2i_model_id,
                torch_dtype=torch.float16,
                use_safetensors=True,
                variant="fp16"
            )
            _t2i_pipeline.to("cuda")
        print(f"Generating image for prompt: '{prompt}'")
        return _t2i_pipeline(prompt).images[0]
    elif mode == "video":
        if _t2v_pipeline is None:
            print("Loading Text-to-Video pipeline...")
            t2v_model_id = 'damo-vilab/text-to-video-ms-1.7b'
            _t2v_pipeline = DiffusionPipeline.from_pretrained(
                t2v_model_id,
                torch_dtype=torch.float16,
                variant="fp16"
            )
            _t2v_pipeline.scheduler = DPMSolverMultistepScheduler.from_config(_t2v_pipeline.scheduler.config)
            _t2v_pipeline.to("cuda")
        print(f"Generating video for prompt: '{prompt}'")
        # Use a reasonable number of frames, e.g., 8-16, for demo purposes
        return _t2v_pipeline(prompt, num_frames=16).frames
    else:
        raise ValueError("Invalid mode. Must be 'image' or 'video'.")

def llm_generate(prompt, max_new_tokens=64, temperature=0.7):
    global tokenizer, model
    if tokenizer is None or model is None:
        raise RuntimeError("LLM tokenizer or model not loaded.")

    # Apply chat template for instruction-tuned models
    messages = [
        {"role": "user", "content": prompt}
    ]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

    # Generate output
    outputs = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id # Important for chat models
    )

    # Decode and return only the new generated text
    response_text = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return response_text

In [ ]:
import json # Ensure json is imported at the top for clarity and use in parsing
import re # Ensure re is imported
import textwrap # Ensure textwrap is imported

def classify_prompt(prompt: str):
    """Classify the user prompt into QA, image, or video."""

    # Step 1: Define a system prompt explaining how to classify requests (qa, image, video)
    # Changed to request JSON output for better parsing, using .format() to avoid f-string issues with dedent and nested {{}}
    system_prompt_template = textwrap.dedent("""
        You are an AI assistant that classifies user requests into one of three categories:
        'qa': For general question-answering.
        'image': For requests to generate an image.
        'video': For requests to generate a video.

        You should also generate an 'expanded_prompt' that is a more detailed version of the user's request, suitable for a generation model.

        Respond with a JSON object containing 'type' and 'expanded_prompt' keys. The 'type' must be one of 'qa', 'image', or 'video'.
        DO NOT include any other text or explanation. Only the JSON object.

        Example 1:
        {{"type": "qa", "expanded_prompt": "Tell me about large language models, their applications, and recent advancements."}}

        Example 2:
        {{"type": "image", "expanded_prompt": "A majestic lion in a savanna at sunset, detailed and realistic."}}

        Example 3:
        {{"type": "video", "expanded_prompt": "A short, high-quality video clip of a car driving through a snowy forest, with snowflakes falling and the car's engine roaring."}}

        User request: {user_request_placeholder}
        Output JSON:
    """).strip()

    system_prompt = system_prompt_template.format(user_request_placeholder=prompt)

    # Step 2 & 3: Format the user message and system message as input to the LLM and generate a response
    # Increased max_new_tokens to allow for full JSON output
    llm_response = llm_generate(system_prompt, max_new_tokens=256, temperature=0.1)

    # Step 4: Robustly parse the response for JSON
    try:
        parsed_data = None

        # First, try to find a JSON block wrapped in ```json
        json_match_block = re.search(r"```json\n(.*?)\n```", llm_response, re.DOTALL)
        if json_match_block:
            try:
                parsed_data = json.loads(json_match_block.group(1))
            except json.JSONDecodeError:
                pass # Continue to broader search if block is malformed

        # If not found or malformed block, search for any JSON object
        if parsed_data is None:
            # Find all potential JSON objects and try to parse the last one (most likely the actual response)
            json_matches = re.findall(r'\{.*?\}', llm_response, re.DOTALL)
            for json_str_candidate in json_matches[::-1]: # Iterate backwards
                try:
                    temp_data = json.loads(json_str_candidate)
                    if "type" in temp_data and "expanded_prompt" in temp_data:
                        parsed_data = temp_data
                        break
                except json.JSONDecodeError:
                    continue # Not a valid JSON object, try next

        if parsed_data:
            request_type = parsed_data.get("type")
            expanded_prompt = parsed_data.get("expanded_prompt")

            if request_type in ["qa", "image", "video"] and expanded_prompt is not None:
                return {"type": request_type, "expanded_prompt": expanded_prompt}
            else:
                print(f"Warning: JSON parsed but invalid content for classification: {parsed_data}")
                # Fallback to QA with original prompt if JSON is malformed or content is unexpected
                return {"type": "qa", "expanded_prompt": prompt}
        else:
            print(f"Warning: No valid JSON object with 'type' and 'expanded_prompt' found in LLM response: {llm_response}")
            return {"type": "qa", "expanded_prompt": prompt}

    except Exception as e:
        print(f"Warning: An error occurred during LLM response parsing for: {llm_response}. Error: {e}")
        # Default to QA if parsing fails or any other unexpected error
        return {"type": "qa", "expanded_prompt": prompt}

### 4.3: Build the multimodal agent
This agent takes a single user prompt, sends it to the `classify_prompt` to determine what kind of task it is, and then calls the appropriate module:
- QA: use the chat LLM to generate an answer
- Image: use the text-to-image generator
- Video: use the text-to-video generator

Start with a simple version first. You can improve it later by adding better prompts, guardrails, and citation handling.

In [ ]:
def multimodal_agent(user_prompt: str):
    # Step 1: Classify the request
    classification = classify_prompt(user_prompt)
    request_type = classification["type"]
    expanded_prompt = classification["expanded_prompt"]

    # Step 2: Route the prompt and generate output
    if request_type == "qa":
        print(f"Routing to QA for prompt: '{expanded_prompt}'")
        return llm_generate(expanded_prompt)
    elif request_type == "image":
        print(f"Routing to Image Generation for prompt: '{expanded_prompt}'")
        return generate_media(expanded_prompt, mode="image")
    elif request_type == "video":
        print(f"Routing to Video Generation for prompt: '{expanded_prompt}'")
        return generate_media(expanded_prompt, mode="video")
    else:
        raise ValueError(f"Unknown request type: {request_type}")

### 4.4: Test the agent
Now let's test your multimodal agent end to end. Each prompt will automatically be routed to the correct capability: text Q&A, image generation, or video generation, and display the corresponding output.

As we observed, all requests were indeed incorrectly routed to video generation. This behavior stems from the TinyLlama model struggling to precisely follow the instructions for classification and JSON output, which is crucial for the classify_prompt function. It's a common challenge with smaller, less-tuned models when complex instruction-following is required. To achieve more accurate routing, you might need to either fine-tune a model for this specific classification task or, as the notebook suggests, consider using a more capable model for the routing component.

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import display, Video
import numpy as np # Import numpy

# Step 1: Define a few diverse prompts (QA, image, video)
prompts = [
    "What is the capital of France?",
    "Generate an image of a majestic lion in a savanna at sunset.",
    "Create a video of a car driving through a snowy forest."
]

# Step 2: For each prompt, call multimodal_agent and inspect the returned result
for i, prompt in enumerate(prompts):
    print(f"\n--- Processing Prompt {i+1}: '{prompt}' ---")
    result = multimodal_agent(prompt)

    if isinstance(result, str):
        print(f"Agent Response (QA):\n{result}")
    elif hasattr(result, 'save'): # PIL Image object
        print("Agent Response (Image):")
        display(result)
        result.save(f"output_image_{i+1}.png")
    elif isinstance(result, np.ndarray): # Corrected: Check for numpy array for video frames
        print("Agent Response (Video):")
        # Ensure the frames are in the correct format (batch, num_frames, H, W, C) or just (num_frames, H, W, C)
        # The export_to_video function expects (num_frames, H, W, C)
        # The result from generate_media for video is a numpy array of shape (1, num_frames, H, W, C), so take the first element.
        video_path = export_to_video(result[0]) # Pass the actual video frames to export_to_video
        display(Video(video_path, embed=True))
        print(f"Video saved to: {video_path}")
    else:
        print(f"Unexpected agent response type: {type(result)}")

Replace the sample queries with your own and verify that the agent chooses the correct generation path.

## 5 - Interactive Web UI

Launch a simple Gradio web interface so you (or your users) can play with the multimodal agent from the browser.


In [ ]:
import gradio as gr
with gr.Blocks() as demo:
    gr.Markdown('# Multimodal Agent')
    inp = gr.Textbox(placeholder='Ask or create...')
    btn = gr.Button('Submit')
    out_text = gr.Markdown()
    out_img = gr.Image()
    out_vid = gr.Video()

    def handle(prompt):
        res = multimodal_agent(prompt)
        if isinstance(res, str):
            return res, None, None
        elif hasattr(res, 'save'):
            return '', res, None
        else:
            vid = export_to_video(res)
            return '', None, vid

    btn.click(handle, inp, [out_text, out_img, out_vid])

demo.launch()

After the UI launches, open the link and generate your own images and videos directly from the browser.

## 🎉 Congratulations!

* You have built a **multi-modal agent** capable of understanding various requests, and routing them to the proper model.
* Try experimenting with other T2I and T2V models.
* Try making your system more efficient. For example, load a separate lightweight llm for routing, and a more capable llm for QA.


👏 **Great job!** Take a moment to celebrate. The techniques you implemented here power many production agents and chatbots.